# SilComputers Production Case

In [2]:
from pulp import LpProblem, LpMaximize, LpVariable, PULP_CBC_CMD, LpStatus, value

def solve_production(memory=15000, assembly=25000):
    model = LpProblem("SilComputers", LpMaximize)
    N = LpVariable("notebooks", lowBound=0)
    D = LpVariable("desktops", lowBound=0)
    model += 750 * N + 1000 * D, "profit"
    model += N + D <= 10000, "processing_chips"
    model += N + 2 * D <= memory, "memory_sets"
    model += 4 * N + 3 * D <= assembly, "assembly_minutes"
    model.solve(PULP_CBC_CMD(msg=0))
    if LpStatus[model.status] != "Optimal":
        raise RuntimeError("An optimal solution was not found.")
    return model, N, D

In [3]:
model, N, D = solve_production()
print(f"Notebooks: {N.value():,.0f}")
print(f"Desktops: {D.value():,.0f}")
print(f"Maximum quarterly profit: ${value(model.objective):,.0f}")

Notebooks: 1,000
Desktops: 7,000
Maximum quarterly profit: $7,750,000


In [4]:
memory_price = model.constraints["memory_sets"].pi
print(f"Value of an extra memory chip set: ${memory_price:,.0f}")

Value of an extra memory chip set: $350


In [5]:
reduced_model, reduced_N, reduced_D = solve_production(assembly=24000)
print(f"Notebooks: {reduced_N.value():,.0f}")
print(f"Desktops: {reduced_D.value():,.0f}")
print(f"Maximum quarterly profit: ${value(reduced_model.objective):,.0f}")
print(f"Profit loss: ${value(model.objective) - value(reduced_model.objective):,.0f}")

Notebooks: 600
Desktops: 7,200
Maximum quarterly profit: $7,650,000
Profit loss: $100,000


In [6]:
chip_price = model.constraints["processing_chips"].pi
assembly_price = model.constraints["assembly_minutes"].pi
required_profit = chip_price + 2 * memory_price + 4 * assembly_price
print(f"Required profit per 32MB notebook: ${required_profit:,.0f}")

Required profit per 32MB notebook: $1,100


Produce 1,000 notebooks and 7,000 desktops. Maximum quarterly profit: **$7,750,000**.

An extra memory chip set is worth **$350** in the LP model. Losing 1,000 assembly minutes reduces profit by **$100,000**, with production of 600 notebooks and 7,200 desktops.

A 32MB notebook requires **$1,100 profit per unit** to break even, assuming four assembly minutes per unit.